# EXP007 — Second-Degree Node Signal Quality

**Question:** Which OSINT signals most reliably predict that a second-degree node is relevant to a given client goal?
**Hypothesis:** Combined embedding_sim + employer_match will outperform individual signals — no single signal is sufficient.
**Success criterion:** Winning signal achieves precision ≥ 0.70 and recall ≥ 0.60 on the 20-node labeled sample.
**Production impact:** `netweave/src/expand.py` → second-degree relevance scoring function (v2).
**Author:** Chuck Wiley  **Date:** 2026-06-23

**Setup required:** Create `experiments/EXP007_second_degree/second_degree_sample.json` with 20 manually labeled second-degree nodes. Estimated labeling time: 2–3 hours.

Format: `[{"node_id": "n001", "relevant": true, "position": "...", "employer": "...", "domain_tags": [...]}]`

In [ ]:
import sys
sys.path.insert(0, ".")

import json
import os
import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from lib.niw_graph import load_graph
from lib.niw_mlflow import compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP007"
EXPERIMENT_NAME = "Second Degree Signal Quality"
mlflow.set_experiment(EXPERIMENT_NAME)

GOAL_TAGS = ["medical_diagnostics", "biotech", "investor"]
GOAL_QUERY = "Find MASLD investors and clinical partners"

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Load manually labeled second-degree sample
SAMPLE_PATH = "experiments/EXP007_second_degree/second_degree_sample.json"
if not os.path.exists(SAMPLE_PATH):
    raise FileNotFoundError(
        f"{SAMPLE_PATH} not found. "
        "Manually label 20 second-degree nodes before running. "
        "Format: [{\"node_id\": \"n001\", \"relevant\": true, \"position\": \"...\", \"employer\": \"...\"}]"
    )

with open(SAMPLE_PATH) as f:
    sample = json.load(f)

print(f"Sample loaded: {len(sample)} nodes ({sum(1 for s in sample if s['relevant'])} relevant)")

encoder = SentenceTransformer("all-MiniLM-L6-v2")
goal_embedding = encoder.encode([GOAL_QUERY]).astype("float32")

In [ ]:
GOAL_TAG_SET = set(GOAL_TAGS)
KEYWORDS = ["bio", "pharma", "genomic", "clinical", "diagnostic", "invest", "venture", "capital", "fund"]

def keyword_match(position: str) -> float:
    text = position.lower()
    return sum(1 for k in KEYWORDS if k in text) / len(KEYWORDS)

def jaccard_employer(node: dict) -> float:
    tags = set(node.get("domain_tags", []))
    if not tags or not GOAL_TAG_SET:
        return 0.0
    return len(tags & GOAL_TAG_SET) / len(tags | GOAL_TAG_SET)

def patent_signal(node: dict) -> float:
    return float(bool(node.get("patent_count", 0)))

def grant_signal(node: dict) -> float:
    return float(bool(node.get("sbir_grant", False)))

def embedding_sim(node: dict) -> float:
    text = f"{node.get('employer', '')} {node.get('position', '')} {' '.join(node.get('domain_tags', []))}"
    emb = encoder.encode([text]).astype("float32")
    return float(cosine_similarity(emb, goal_embedding)[0][0])

def combined_signal(node: dict) -> float:
    return (
        0.35 * embedding_sim(node)
        + 0.30 * jaccard_employer(node)
        + 0.20 * keyword_match(node.get("position", ""))
        + 0.10 * patent_signal(node)
        + 0.05 * grant_signal(node)
    )

signal_extractors = {
    "title_keyword":  lambda node: keyword_match(node.get("position", "")),
    "employer_match": lambda node: jaccard_employer(node),
    "patent_signal":  lambda node: patent_signal(node),
    "grant_signal":   lambda node: grant_signal(node),
    "embedding_sim":  lambda node: embedding_sim(node),
    "combined":       lambda node: combined_signal(node),
}

In [ ]:
y_true = [1 if s["relevant"] else 0 for s in sample]

results = []
for signal_name, extractor in signal_extractors.items():
    with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{signal_name}"):
        mlflow.log_params({"signal": signal_name})

        scores = [extractor(node) for node in sample]
        # Threshold at median score
        threshold = float(np.median(scores))
        y_pred = [1 if s >= threshold else 0 for s in scores]

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        mlflow.log_metrics({"precision": prec, "recall": rec, "f1": f1})
        results.append({"variant": signal_name, "precision": prec, "recall": rec, "f1": f1})
        print(f"{signal_name}: precision={prec:.4f}, recall={rec:.4f}, f1={f1:.4f}")

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print(results_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].barh(results_df["variant"], results_df["f1"])
axes[0].set_xlabel("F1 Score")
axes[0].set_title(f"{EXPERIMENT_ID}: Signal Quality (F1)")
axes[0].invert_yaxis()

x = np.arange(len(results_df))
width = 0.35
axes[1].barh(x - width/2, results_df["precision"], width, label="Precision")
axes[1].barh(x + width/2, results_df["recall"], width, label="Recall")
axes[1].set_yticks(x)
axes[1].set_yticklabels(results_df["variant"])
axes[1].set_xlabel("Score")
axes[1].set_title(f"{EXPERIMENT_ID}: Precision vs. Recall")
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("experiments/EXP007_second_degree/results.png", dpi=150)
plt.show()

## Result and Decision

**Winner:** [fill in after running]
**Precision:** [fill in]
**Recall:** [fill in]
**F1:** [fill in]
**Decision:** [VALIDATED | INCONCLUSIVE | REJECTED]

**If VALIDATED — production change:**
File: `netweave/src/expand.py`
Function: Second-degree relevance scoring function (v2 implementation).
Change: Use winning signal combination with validated weights.
Notebook citation: `# Validated in EXP007 — see netweave-lab/experiments/EXP007_second_degree/`